# Evaluating iAmTime on GIFT-Eval

This notebook evaluates the public [`anishsaha/iAmTime-base-forecast`](https://huggingface.co/anishsaha/iAmTime-base-forecast) checkpoint on GIFT-Eval test splits. The default dataset selection is a small smoke test; enable the commented full dataset lists to reproduce the leaderboard submission.

The notebook uses iAmTime's public forecasting pipeline directly. It does not depend on private ICL data-preparation code, local model checkpoints, or machine-specific paths.

Before running the evaluation, download the GIFT-Eval dataset and set `GIFT_EVAL` as described in the [repository README](../README.md).

## Environment and data

Use Python 3.10 or later. From the GIFT-Eval repository root, install the benchmark dependencies:

```bash
pip install -e ".[baseline]"
```

Install iAmTime from a clone of its repository (the project README documents this installation):

```bash
git clone https://github.com/anishsaha12/iAmTime.git
cd iAmTime
pip install .
```

Download the GIFT-Eval dataset and set `GIFT_EVAL` as documented in the [GIFT-Eval README](../README.md). On the first run, `from_pretrained` downloads the public forecasting checkpoint from Hugging Face and caches it locally.

In [5]:
import json
import logging
import os
from pathlib import Path

import numpy as np
import torch
from dotenv import load_dotenv
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.itertools import batcher
from gluonts.model import evaluate_model
from gluonts.model.forecast import QuantileForecast
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset
from iAmTime.pipeline import iAmTimePipeline

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src" / "gift_eval").is_dir():
    raise RuntimeError("Run this notebook from the GIFT-Eval repository root or notebooks/.")

load_dotenv(REPO_ROOT / ".env")
if not os.environ.get("GIFT_EVAL"):
    raise RuntimeError("Set GIFT_EVAL to the downloaded GIFT-Eval dataset directory.")

MODEL_ID = "anishsaha/iAmTime-base-forecast"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float32
BATCH_SIZE = 64
OUTPUT_PATH = REPO_ROOT / "results" / "iAmTime" / "all_results.csv"

print(f"Model: {MODEL_ID}")
print(f"Device: {DEVICE}; dtype: {TORCH_DTYPE}")
print(f"Results: {OUTPUT_PATH}")

Model: anishsaha/iAmTime-base-forecast
Device: cuda; dtype: torch.float32


## Benchmark coverage

The active selectors below mirror the small dataset example in `chronos-2.ipynb`. The complete benchmark lists are retained as comments. Uncomment the full lists and disable the sample assignments to run all configurations; a sample run writes `sample_results.csv` so it cannot replace the validated submission file.

In [6]:
#SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
SHORT_DATASETS = "m4_weekly"

#MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
MED_LONG_DATASETS = "bizitobs_l2c/H"

In [7]:
short_dataset_names = set(SHORT_DATASETS.split())
med_long_dataset_names = set(MED_LONG_DATASETS.split())
all_datasets = sorted(short_dataset_names | med_long_dataset_names)
dataset_properties_map = json.loads(
    (REPO_ROOT / "notebooks" / "dataset_properties.json").read_text()
)

expected_rows = len(short_dataset_names) + 2 * len(med_long_dataset_names)
print(f"Datasets: {len(all_datasets)}")
print(f"Expected evaluation rows: {expected_rows}")

Datasets: 2
Expected evaluation rows: 3


In [8]:
QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=QUANTILE_LEVELS),
]

In [9]:
pipeline = iAmTimePipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=TORCH_DTYPE,
)
pipeline.model.eval()
print("iAmTime forecasting checkpoint loaded.")

iAmTime forecasting checkpoint loaded.


## Forecast predictor

GIFT-Eval's predictor interface expects GluonTS `Forecast` objects. The adapter sends each univariate target history to iAmTime in batches and converts the returned `(batch, horizon, quantiles)` array into `QuantileForecast` objects. The checkpoint is loaded once and reused across all dataset configurations.

Multivariate source datasets are converted to univariate series, matching the reference foundation-model evaluations and the iAmTime checkpoint's per-series forecasting interface.

In [10]:
class IAmTimePredictor:
    def __init__(self, pipeline, prediction_length: int, batch_size: int = BATCH_SIZE):
        self.pipeline = pipeline
        self.prediction_length = prediction_length
        self.batch_size = batch_size

    def _predict_histories(self, histories: list[np.ndarray]) -> np.ndarray:
        try:
            quantiles, _ = self.pipeline.predict_quantiles(
                histories,
                prediction_length=self.prediction_length,
                quantile_levels=QUANTILE_LEVELS,
            )
        except torch.cuda.OutOfMemoryError:
            if len(histories) == 1:
                raise
            torch.cuda.empty_cache()
            midpoint = len(histories) // 2
            return np.concatenate(
                [
                    self._predict_histories(histories[:midpoint]),
                    self._predict_histories(histories[midpoint:]),
                ],
                axis=0,
            )

        if isinstance(quantiles, torch.Tensor):
            quantiles = quantiles.detach().cpu().numpy()
        quantiles = np.asarray(quantiles)
        expected_shape = (
            len(histories),
            self.prediction_length,
            len(QUANTILE_LEVELS),
        )
        if quantiles.shape != expected_shape:
            raise ValueError(
                f"Expected quantiles with shape {expected_shape}, got {quantiles.shape}."
            )
        return quantiles

    def predict(self, test_data_input, **kwargs):
        forecasts = []
        for batch in batcher(test_data_input, batch_size=self.batch_size):
            targets = [np.asarray(item["target"]) for item in batch]
            if any(target.ndim != 1 for target in targets):
                raise ValueError("Expected univariate target histories after dataset conversion.")

            batch_quantiles = self._predict_histories(targets)
            for item, target, quantiles in zip(batch, targets, batch_quantiles):
                forecasts.append(
                    QuantileForecast(
                        forecast_arrays=quantiles.T,
                        forecast_keys=list(map(str, QUANTILE_LEVELS)),
                        start_date=item["start"] + len(target),
                    )
                )
        return forecasts

## Evaluate and write results

For each selected dataset and horizon, the notebook evaluates with `evaluate_model` using the benchmark's official metrics, `axis=None`, `mask_invalid_label=True`, `allow_nan_forecast=False`, and frequency-specific seasonality. The output is the leaderboard's 15-column CSV format, written to `results/iAmTime/all_results.csv`.

In [11]:
import logging


class WarningFilter(logging.Filter):
    def __init__(self, text_to_filter):
        super().__init__()
        self.text_to_filter = text_to_filter

    def filter(self, record):
        return self.text_to_filter not in record.getMessage()


gts_logger = logging.getLogger("gluonts.model.forecast")
gts_logger.addFilter(
    WarningFilter("The mean prediction is not stored in the forecast data")
)

In [14]:
import pandas as pd

pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

CSV_COLUMNS = [
    "dataset",
    "model",
    "eval_metrics/MSE[mean]",
    "eval_metrics/MSE[0.5]",
    "eval_metrics/MAE[0.5]",
    "eval_metrics/MASE[0.5]",
    "eval_metrics/MAPE[0.5]",
    "eval_metrics/sMAPE[0.5]",
    "eval_metrics/MSIS",
    "eval_metrics/RMSE[mean]",
    "eval_metrics/NRMSE[mean]",
    "eval_metrics/ND[0.5]",
    "eval_metrics/mean_weighted_sum_quantile_loss",
    "domain",
    "num_variates",
]

results_rows = []
for dataset_index, dataset_name in enumerate(all_datasets, start=1):
    print(f"[{dataset_index}/{len(all_datasets)}] {dataset_name}")
    terms = []
    if dataset_name in short_dataset_names:
        terms.append("short")
    if dataset_name in med_long_dataset_names:
        terms.extend(("medium", "long"))

    for term in terms:
        source_dataset = Dataset(name=dataset_name, term=term, to_univariate=False)
        dataset = Dataset(
            name=dataset_name,
            term=term,
            to_univariate=source_dataset.target_dim > 1,
        )

        dataset_key = dataset_name.split("/")[0].lower()
        dataset_key = pretty_names.get(dataset_key, dataset_key)
        if "/" in dataset_name:
            frequency = dataset_name.split("/")[1]
        else:
            frequency = dataset_properties_map[dataset_key]["frequency"]
        dataset_config = f"{dataset_key}/{frequency}/{term}"

        predictor = IAmTimePredictor(
            pipeline=pipeline,
            prediction_length=dataset.prediction_length,
        )
        evaluation = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=512,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=get_seasonality(dataset.freq),
        )

        results_rows.append(
            {
                "dataset": dataset_config,
                "model": "iAmTime",
                "eval_metrics/MSE[mean]": float(evaluation["MSE[mean]"][0]),
                "eval_metrics/MSE[0.5]": float(evaluation["MSE[0.5]"][0]),
                "eval_metrics/MAE[0.5]": float(evaluation["MAE[0.5]"][0]),
                "eval_metrics/MASE[0.5]": float(evaluation["MASE[0.5]"][0]),
                "eval_metrics/MAPE[0.5]": float(evaluation["MAPE[0.5]"][0]),
                "eval_metrics/sMAPE[0.5]": float(evaluation["sMAPE[0.5]"][0]),
                "eval_metrics/MSIS": float(evaluation["MSIS"][0]),
                "eval_metrics/RMSE[mean]": float(evaluation["RMSE[mean]"][0]),
                "eval_metrics/NRMSE[mean]": float(evaluation["NRMSE[mean]"][0]),
                "eval_metrics/ND[0.5]": float(evaluation["ND[0.5]"][0]),
                "eval_metrics/mean_weighted_sum_quantile_loss": float(
                    evaluation["mean_weighted_sum_quantile_loss"][0]
                ),
                "domain": dataset_properties_map[dataset_key]["domain"],
                "num_variates": dataset_properties_map[dataset_key]["num_variates"],
            }
        )

results_df = pd.DataFrame(results_rows, columns=CSV_COLUMNS).sort_values("dataset")
if len(results_df) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} selected configurations, got {len(results_df)}."
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
result_path = (
    OUTPUT_PATH
    if expected_rows == 97
    else OUTPUT_PATH.with_name("sample_results.csv")
)
results_df.to_csv(result_path, index=False)
print(f"Wrote {len(results_df)} dataset results to {result_path}")

[1/2] bizitobs_l2c/H
7it [00:00, 57.82it/s]
7it [00:00, 11.47it/s]
[2/2] m4_weekly
359it [00:00, 450.62it/s]
Wrote 3 dataset results to /home/jupyter/notebooks/results/iAmTime/sample_results.csv


## Submission artifact

A complete run writes all 97 benchmark configurations and 11 metrics to `results/iAmTime/all_results.csv`, alongside the existing `config.json`. The default sample run writes `results/iAmTime/sample_results.csv` instead.